### 1. Ingest all raw file into the Bronze Layer


In [0]:
# List of all your uploaded files
file_names = [
    "customers",
    "employees",
    "order_items",
    "orders",
    "products",
    "stores",
]

from pyspark.sql.functions import current_timestamp, lit

for file_name in file_names:
  # Construct the volume file path for each CSV
  file_path = (
      f"dbfs:/Volumes/walmart_project_catalog/bronze/raw_data/{file_name}.csv"
  )

  # Read raw CSV into Spark DataFrame
  raw_df = (
      spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(file_path)
  )

  # Add auditing metadata columns (Fixed withColumn capital C)
  bronze_df = (
      raw_df.withColumn("ingestion_timestamp", current_timestamp()).withColumn(
          "source_file", lit(f"{file_name}.csv")
      )
  )

  # Save as a Delta table in the bronze schema
  table_name = f"walmart_project_catalog.bronze.bronze_{file_name}"
  bronze_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

  print(f"Successfully created Bronze table: {table_name}")

print("\nAll Bronze tables created successfully!")

Successfully created Bronze table: walmart_project_catalog.bronze.bronze_customers
Successfully created Bronze table: walmart_project_catalog.bronze.bronze_employees
Successfully created Bronze table: walmart_project_catalog.bronze.bronze_order_items
Successfully created Bronze table: walmart_project_catalog.bronze.bronze_orders
Successfully created Bronze table: walmart_project_catalog.bronze.bronze_products
Successfully created Bronze table: walmart_project_catalog.bronze.bronze_stores

All Bronze tables created successfully!


In [0]:
# List all tables inside your bronze schema
spark.sql("SHOW TABLES IN walmart_project_catalog.bronze").display()

database,tableName,isTemporary
bronze,bronze_customers,false
bronze,bronze_employees,false
bronze,bronze_order_items,false
bronze,bronze_orders,false
bronze,bronze_products,false
bronze,bronze_stores,false


### 2. Clean and Transform store to Silver Layer


In [0]:
# Read from the bronze stores table
bronze_stores_df = spark.table("walmart_project_catalog.bronze.bronze_stores")

# Clean the data: drop duplicates
silver_stores_df = bronze_stores_df.dropDuplicates()

# Save as a Delta table in the silver schema
(
    silver_stores_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("walmart_project_catalog.silver.silver_stores")
)

print("Silver Stores table created successfully!")
display(spark.table("walmart_project_catalog.silver.silver_stores").limit(5))

Silver Stores table created successfully!


store_id,store_name,city,province,country,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
11,Walmart Store 11,Mitchellton,Kansas,Canada,2026-02-25T19:31:19.000Z,2026-03-15T20:08:46.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
15,Walmart Store 15,Smithshire,Hawaii,Canada,2026-05-17T07:21:17.000Z,2026-06-08T00:53:36.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
5,Walmart Store 5,North Sabrinamouth,Louisiana,Canada,2026-01-21T01:28:36.000Z,2026-02-06T08:43:52.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
14,Walmart Store 14,Mollychester,Oregon,Canada,2026-02-15T10:32:30.000Z,2026-03-03T20:13:49.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
25,Walmart Store 25,East Raymond,Kentucky,Canada,2026-01-15T13:22:41.000Z,2026-02-09T17:52:30.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv


#### Trans- 1. Silver Layer Transformation: Cleaning Store Geography


In [0]:
from pyspark.sql.functions import col, when

# Read from bronze stores
bronze_stores_df = spark.table("walmart_project_catalog.bronze.bronze_stores")

# Clean and transform: Fix the country/province mapping mismatch
# Since these are US states, let's correct 'Canada' to 'United States' for these rows
silver_stores_df = bronze_stores_df.dropDuplicates().withColumn(
    "country",
    when(
        col("province").isin(
            "Kansas", "Hawaii", "Louisiana", "Oregon", "Kentucky"
        ),
        "United States",
    ).otherwise(col("country")),
)

# Save as a Delta table in the silver schema
(
    silver_stores_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("walmart_project_catalog.silver.silver_stores")
)

print("Silver Stores table cleaned and created successfully!")
display(spark.table("walmart_project_catalog.silver.silver_stores").limit(5))

Silver Stores table cleaned and created successfully!


store_id,store_name,city,province,country,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
11,Walmart Store 11,Mitchellton,Kansas,United States,2026-02-25T19:31:19.000Z,2026-03-15T20:08:46.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
15,Walmart Store 15,Smithshire,Hawaii,United States,2026-05-17T07:21:17.000Z,2026-06-08T00:53:36.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
5,Walmart Store 5,North Sabrinamouth,Louisiana,United States,2026-01-21T01:28:36.000Z,2026-02-06T08:43:52.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
14,Walmart Store 14,Mollychester,Oregon,United States,2026-02-15T10:32:30.000Z,2026-03-03T20:13:49.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
25,Walmart Store 25,East Raymond,Kentucky,United States,2026-01-15T13:22:41.000Z,2026-02-09T17:52:30.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv


#### Silver Layer Transformation: 2. Fixing Store Geography (City and Country)

In [0]:
from pyspark.sql.functions import col, when

# Read from bronze stores
bronze_stores_df = spark.table("walmart_project_catalog.bronze.bronze_stores")

# Clean and correct mismatched cities, provinces, and countries
silver_stores_df = bronze_stores_df.dropDuplicates().withColumn(
    "city",
    when(col("store_id") == 11, "Wichita")
    .when(col("store_id") == 15, "Honolulu")
    .when(col("store_id") == 5, "New Orleans")
    .when(col("store_id") == 14, "Portland")
    .when(col("store_id") == 25, "Louisville")
    .otherwise(col("city")),
).withColumn(
    "country", lit("United States")
)  # Standardize all to United States since these are US states

# Save as a Delta table in the silver schema
(
    silver_stores_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("walmart_project_catalog.silver.silver_stores")
)

print("Silver Stores table corrected successfully!")
display(spark.table("walmart_project_catalog.silver.silver_stores"))

Silver Stores table corrected successfully!


store_id,store_name,city,province,country,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
11,Walmart Store 11,Wichita,Kansas,United States,2026-02-25T19:31:19.000Z,2026-03-15T20:08:46.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
15,Walmart Store 15,Honolulu,Hawaii,United States,2026-05-17T07:21:17.000Z,2026-06-08T00:53:36.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
5,Walmart Store 5,New Orleans,Louisiana,United States,2026-01-21T01:28:36.000Z,2026-02-06T08:43:52.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
14,Walmart Store 14,Portland,Oregon,United States,2026-02-15T10:32:30.000Z,2026-03-03T20:13:49.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
25,Walmart Store 25,Louisville,Kentucky,United States,2026-01-15T13:22:41.000Z,2026-02-09T17:52:30.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
12,Walmart Store 12,East Vincentfurt,Maine,United States,2026-01-29T23:56:28.000Z,2026-02-23T20:09:05.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
21,Walmart Store 21,Sarahland,Maryland,United States,2026-06-06T21:10:20.000Z,2026-07-07T13:41:01.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
1,Walmart Store 1,South Nicolestad,Illinois,United States,2026-02-22T07:09:17.000Z,2026-03-14T06:59:24.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
3,Walmart Store 3,Jeanettemouth,Rhode Island,United States,2026-05-15T07:47:12.000Z,2026-05-27T04:49:37.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
13,Walmart Store 13,Chrisstad,South Carolina,United States,2026-06-07T14:00:16.000Z,2026-07-02T03:38:33.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv


In [0]:
# List of all your bronze tables to preview
tables = ["customers", "employees", "order_items", "orders", "products", "stores"]

for t in tables:
  print(f"--- Preview of Bronze Table: bronze_{t} ---")
  display(spark.table(f"walmart_project_catalog.bronze.bronze_{t}").limit(3))

--- Preview of Bronze Table: bronze_customers ---


customer_id,first_name,last_name,email,phone,city,province,country,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
1,Ronald,Long,shall@example.com,+1-372-617-6276x55955,Torresland,Alaska,Canada,2026-06-13T03:01:47.000Z,2026-06-21T10:15:55.000Z,Y,2026-08-30T18:49:37.288Z,customers.csv
2,Carlos,Zimmerman,cesar33@example.org,4587577121,Lake Whitney,California,Canada,2026-06-23T23:57:34.000Z,2026-06-26T18:24:36.000Z,Y,2026-08-30T18:49:37.288Z,customers.csv
3,Austin,Tucker,tinamcdaniel@example.net,+1-978-494-3397x331,Richardsonburgh,New Jersey,Canada,2026-02-25T07:32:38.000Z,2026-02-26T00:45:23.000Z,Y,2026-08-30T18:49:37.288Z,customers.csv


--- Preview of Bronze Table: bronze_employees ---


employee_id,store_id,first_name,last_name,email,job_title,salary,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
1,9,Robert,Diaz,angelarussell@example.com,IT consultant,52919.01,2026-05-30T16:26:50.000Z,2026-06-24T07:01:18.000Z,Y,2026-08-30T18:49:45.805Z,employees.csv
2,9,Alyssa,Pitts,omarthomas@example.com,Landscape architect,67480.97,2026-05-07T22:21:32.000Z,2026-05-18T22:19:14.000Z,Y,2026-08-30T18:49:45.805Z,employees.csv
3,20,Megan,Garcia,brooke90@example.net,Hydrographic surveyor,52281.96,2026-05-31T13:23:50.000Z,2026-06-28T00:14:35.000Z,Y,2026-08-30T18:49:45.805Z,employees.csv


--- Preview of Bronze Table: bronze_order_items ---


...

order_id,customer_id,store_id,order_timestamp,payment_method,order_status,total_amount,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
1,522,10,2026-12-02T01:28:27.000Z,Cash,Pending,4202.65,2026-04-11T06:43:19.000Z,2026-04-30T13:28:33.000Z,Y,2026-08-30T18:49:56.276Z,orders.csv
2,1475,10,2026-08-24T05:56:23.000Z,Cash,Cancelled,595.47,2026-06-20T01:23:34.000Z,2026-06-27T08:38:50.000Z,Y,2026-08-30T18:49:56.276Z,orders.csv
3,929,13,2026-10-28T22:57:46.000Z,Cash,Pending,2637.61,2026-03-10T23:49:57.000Z,2026-04-09T20:11:09.000Z,Y,2026-08-30T18:49:56.276Z,orders.csv


--- Preview of Bronze Table: bronze_products ---


product_id,product_name,category,brand,price,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
1,After Product,Home,Samsung,164.31,2026-06-30T06:13:42.000Z,2026-07-22T14:32:35.000Z,Y,2026-08-30T18:50:01.482Z,products.csv
2,Participant Product,Sports,Adidas,140.61,2026-03-25T17:35:47.000Z,2026-04-01T00:15:19.000Z,Y,2026-08-30T18:50:01.482Z,products.csv
3,Use Product,Electronics,Adidas,331.41,2026-02-11T10:46:30.000Z,2026-03-14T10:24:00.000Z,Y,2026-08-30T18:50:01.482Z,products.csv


--- Preview of Bronze Table: bronze_stores ---


store_id,store_name,city,province,country,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
1,Walmart Store 1,South Nicolestad,Illinois,Canada,2026-02-22T07:09:17.000Z,2026-03-14T06:59:24.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
2,Walmart Store 2,New Williamfurt,Utah,Canada,2026-04-02T15:29:36.000Z,2026-05-02T12:27:36.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv
3,Walmart Store 3,Jeanettemouth,Rhode Island,Canada,2026-05-15T07:47:12.000Z,2026-05-27T04:49:37.000Z,Y,2026-08-30T18:50:05.812Z,stores.csv


####Silver Layer Transformation: Employees and Products

In [0]:
# List of remaining tables to clean and move to Silver
silver_tables = ["customers", "employees", "order_items", "orders", "products"]

for table in silver_tables:
  # Read from Bronze
  source_table = f"walmart_project_catalog.bronze.bronze_{table}"
  df = spark.table(source_table)

  # Clean data: drop duplicates
  cleaned_df = df.dropDuplicates()

  # Save to Silver schema
  target_table = f"walmart_project_catalog.silver.silver_{table}"
  cleaned_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

  print(f"Successfully created Silver table: {target_table}")

print("\nAll remaining Silver tables created successfully!")

Successfully created Silver table: walmart_project_catalog.silver.silver_customers
Successfully created Silver table: walmart_project_catalog.silver.silver_employees
Successfully created Silver table: walmart_project_catalog.silver.silver_order_items
Successfully created Silver table: walmart_project_catalog.silver.silver_orders
Successfully created Silver table: walmart_project_catalog.silver.silver_products

All remaining Silver tables created successfully!


#### Silver Layer Transformation: Orders and Order Items


In [0]:
# Clean and transform Orders
bronze_orders_df = spark.table("walmart_project_catalog.bronze.bronze_orders")
silver_orders_df = bronze_orders_df.dropDuplicates()

silver_orders_df.write.format("delta").mode("overwrite").saveAsTable(
    "walmart_project_catalog.silver.silver_orders"
)
print("Silver Orders table created successfully!")

# Clean and transform Order Items
bronze_order_items_df = spark.table(
    "walmart_project_catalog.bronze.bronze_order_items"
)
silver_order_items_df = bronze_order_items_df.dropDuplicates()

silver_order_items_df.write.format("delta").mode("overwrite").saveAsTable(
    "walmart_project_catalog.silver.silver_order_items"
)
print("Silver Order Items table created successfully!")

# Preview the results
display(spark.table("walmart_project_catalog.silver.silver_orders").limit(3))
display(
    spark.table("walmart_project_catalog.silver.silver_order_items").limit(3)
)

Silver Orders table created successfully!
Silver Order Items table created successfully!


order_id,customer_id,store_id,order_timestamp,payment_method,order_status,total_amount,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
3,929,13,2026-10-28T22:57:46.000Z,Cash,Pending,2637.61,2026-03-10T23:49:57.000Z,2026-04-09T20:11:09.000Z,Y,2026-08-30T18:49:56.276Z,orders.csv
45,531,22,2026-05-19T14:07:33.000Z,Online,Completed,2278.49,2026-05-29T07:37:20.000Z,2026-06-23T14:36:21.000Z,Y,2026-08-30T18:49:56.276Z,orders.csv
54,1560,7,2026-01-02T01:01:01.000Z,Credit Card,Cancelled,3035.04,2026-05-21T16:24:15.000Z,2026-06-16T02:12:00.000Z,Y,2026-08-30T18:49:56.276Z,orders.csv


order_item_id,order_id,product_id,quantity,unit_price,line_amount,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
22,8,90,1,147.23,147.23,2026-05-02T22:44:53.000Z,2026-05-31T06:35:15.000Z,Y,2026-08-30T18:49:51.493Z,order_items.csv
65,20,212,1,313.04,313.04,2026-06-03T05:59:16.000Z,2026-06-04T18:22:56.000Z,Y,2026-08-30T18:49:51.493Z,order_items.csv
73,22,227,1,155.84,155.84,2026-05-13T15:32:38.000Z,2026-06-01T01:32:55.000Z,Y,2026-08-30T18:49:51.493Z,order_items.csv


In [0]:
from pyspark.sql.functions import col, round

# Read from silver order_items
silver_order_items_df = spark.table(
    "walmart_project_catalog.silver.silver_order_items"
)

# Round monetary columns to 2 decimal places (adjust column name if yours is 'unit_price' or 'price')
cleaned_order_items_df = silver_order_items_df.withColumn(
    "unit_price", round(col("unit_price"), 2)
)

# Overwrite the Silver table with the rounded values
(
    cleaned_order_items_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("walmart_project_catalog.silver.silver_order_items")
)

print("Silver Order Items rounded to 2 decimal places successfully!")
display(spark.table("walmart_project_catalog.silver.silver_order_items").limit(5))

Silver Order Items rounded to 2 decimal places successfully!


order_item_id,order_id,product_id,quantity,unit_price,line_amount,created_timestamp,updated_timestamp,is_active,ingestion_timestamp,source_file
22,8,90,1,147.23,147.23,2026-05-02T22:44:53.000Z,2026-05-31T06:35:15.000Z,Y,2026-08-30T18:49:51.493Z,order_items.csv
65,20,212,1,313.04,313.04,2026-06-03T05:59:16.000Z,2026-06-04T18:22:56.000Z,Y,2026-08-30T18:49:51.493Z,order_items.csv
73,22,227,1,155.84,155.84,2026-05-13T15:32:38.000Z,2026-06-01T01:32:55.000Z,Y,2026-08-30T18:49:51.493Z,order_items.csv
88,27,32,1,177.18,177.18,2026-02-20T23:10:34.000Z,2026-03-03T05:44:59.000Z,Y,2026-08-30T18:49:51.493Z,order_items.csv
108,33,150,1,281.19,281.19,2026-05-25T06:52:28.000Z,2026-06-12T04:47:25.000Z,N,2026-08-30T18:49:51.493Z,order_items.csv


In [0]:
from pyspark.sql.functions import col, count, round, sum

# Read from Silver tables
stores_df = spark.table("walmart_project_catalog.silver.silver_stores")
orders_df = spark.table("walmart_project_catalog.silver.silver_orders")
order_items_df = spark.table(
    "walmart_project_catalog.silver.silver_order_items"
)

# Join and aggregate with 2-decimal rounding on total revenue
store_sales_df = (
    stores_df.join(orders_df, "store_id")
    .join(order_items_df, "order_id")
    .groupBy("store_id", "store_name", "city", "province")
    .agg(
        count("order_id").alias("total_orders"),
        round(sum(col("quantity") * col("unit_price")), 2).alias(
            "total_revenue"
        ),
    )
)

# Overwrite the Gold table
(
    store_sales_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("walmart_project_catalog.gold.gold_store_sales_summary")
)

print("Gold Store Sales Summary created with rounded revenue!")
display(
    spark.table("walmart_project_catalog.gold.gold_store_sales_summary")
    .orderBy(col("total_revenue").desc())
    .limit(10)
)

Gold Store Sales Summary created with rounded revenue!


store_id,store_name,city,province,total_orders,total_revenue
10,Walmart Store 10,Nixonburgh,Oklahoma,1295,840945.79
23,Walmart Store 23,New Teresa,North Dakota,1321,834146.53
22,Walmart Store 22,Kristaberg,Minnesota,1245,827663.99
13,Walmart Store 13,Chrisstad,South Carolina,1255,815177.17
21,Walmart Store 21,Sarahland,Maryland,1294,814880.66
17,Walmart Store 17,Jasmineburgh,Washington,1288,802285.8
6,Walmart Store 6,New Brandontown,Maryland,1242,780727.26
3,Walmart Store 3,Jeanettemouth,Rhode Island,1229,776718.43
12,Walmart Store 12,East Vincentfurt,Maine,1223,775878.18
4,Walmart Store 4,West Richardville,New Jersey,1192,771296.67


### The Gold Layer (Business Aggreations & Metrics)

In [0]:
# Ensure gold schema exists and drop the conflicting table to allow schema recreation
spark.sql("CREATE SCHEMA IF NOT EXISTS walmart_project_catalog.gold")
spark.sql(
    "DROP TABLE IF EXISTS walmart_project_catalog.gold.gold_store_sales_summary"
)

from pyspark.sql.functions import col, count, sum

# Read from Silver tables
stores_df = spark.table("walmart_project_catalog.silver.silver_stores")
orders_df = spark.table("walmart_project_catalog.silver.silver_orders")
order_items_df = spark.table(
    "walmart_project_catalog.silver.silver_order_items"
)

# Join, aggregate, and cast to decimal(18,2)
store_sales_df = (
    stores_df.join(orders_df, "store_id")
    .join(order_items_df, "order_id")
    .groupBy("store_id", "store_name", "city", "province")
    .agg(
        count("order_id").alias("total_orders"),
        sum(col("quantity") * col("unit_price"))
        .cast("decimal(18,2)")
        .alias("total_revenue"),
    )
)

# Save as a Delta table in the gold schema
(
    store_sales_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("walmart_project_catalog.gold.gold_store_sales_summary")
)

print(
    "Gold Store Sales Summary recreated successfully with 2 decimal precision!"
)
display(
    spark.table("walmart_project_catalog.gold.gold_store_sales_summary")
    .orderBy(col("total_revenue").desc())
    .limit(10)
)

Gold Store Sales Summary recreated successfully with 2 decimal precision!


store_id,store_name,city,province,total_orders,total_revenue
10,Walmart Store 10,Nixonburgh,Oklahoma,1295,840945.79
23,Walmart Store 23,New Teresa,North Dakota,1321,834146.53
22,Walmart Store 22,Kristaberg,Minnesota,1245,827663.99
13,Walmart Store 13,Chrisstad,South Carolina,1255,815177.17
21,Walmart Store 21,Sarahland,Maryland,1294,814880.66
17,Walmart Store 17,Jasmineburgh,Washington,1288,802285.80
6,Walmart Store 6,New Brandontown,Maryland,1242,780727.26
3,Walmart Store 3,Jeanettemouth,Rhode Island,1229,776718.43
12,Walmart Store 12,East Vincentfurt,Maine,1223,775878.18
4,Walmart Store 4,West Richardville,New Jersey,1192,771296.67


### Final Gold Layer Aggregations (Top Products & Customer Summaries)

In [0]:
from pyspark.sql.functions import col, count, sum

# 1. Build Gold Top Products Table
products_df = spark.table("walmart_project_catalog.silver.silver_products")
order_items_df = spark.table(
    "walmart_project_catalog.silver.silver_order_items"
)

top_products_df = (
    products_df.join(order_items_df, "product_id")
    .groupBy("product_id", "product_name", "category")
    .agg(
        sum("quantity").alias("total_units_sold"),
        sum(col("quantity") * col("unit_price"))
        .cast("decimal(18,2)")
        .alias("total_product_revenue"),
    )
)

top_products_df.write.format("delta").mode("overwrite").saveAsTable(
    "walmart_project_catalog.gold.gold_top_products"
)
print("Gold Top Products table created successfully!")


# 2. Build Gold Customer Spending Table
customers_df = spark.table("walmart_project_catalog.silver.silver_customers")
orders_df = spark.table("walmart_project_catalog.silver.silver_orders")
order_items_df = spark.table(
    "walmart_project_catalog.silver.silver_order_items"
)

customer_spending_df = (
    customers_df.join(orders_df, "customer_id")
    .join(order_items_df, "order_id")
    .groupBy("customer_id", "first_name", "last_name", "email")
    .agg(
        count("order_id").alias("total_orders"),
        sum(col("quantity") * col("unit_price"))
        .cast("decimal(18,2)")
        .alias("total_spent"),
    )
)

customer_spending_df.write.format("delta").mode("overwrite").saveAsTable(
    "walmart_project_catalog.gold.gold_customer_spending"
)
print("Gold Customer Spending table created successfully!")

# Display previews
print("\n--- Top Products Preview ---")
display(
    spark.table("walmart_project_catalog.gold.gold_top_products")
    .orderBy(col("total_product_revenue").desc())
    .limit(5)
)

print("\n--- Top Customers Preview ---")
display(
    spark.table("walmart_project_catalog.gold.gold_customer_spending")
    .orderBy(col("total_spent").desc())
    .limit(5)
)

Gold Top Products table created successfully!
Gold Customer Spending table created successfully!

--- Top Products Preview ---


product_id,product_name,category,total_units_sold,total_product_revenue
142,Arm Product,Toys,201,60900.45
29,Left Product,Grocery,220,60838.21
357,Quickly Product,Electronics,225,58573.26
355,East Product,Sports,199,55132.95
466,Board Product,Grocery,211,54114.11



--- Top Customers Preview ---


customer_id,first_name,last_name,email,total_orders,total_spent
1138,Spencer,Barber,richardcampbell@example.net,44,35447.22
925,Vanessa,Juarez,patrickshelton@example.org,51,34157.29
1260,Shannon,Hanson,cindy68@example.com,49,31721.66
532,Colton,Hendrix,edwardscynthia@example.net,35,31708.66
908,Kelly,Ruiz,nathanlevy@example.net,37,30498.99


In [0]:
display(
    spark.sql(
        "SELECT * FROM walmart_project_catalog.gold.gold_store_sales_summary"
        " LIMIT 5"
    )
)

store_id,store_name,city,province,total_orders,total_revenue
12,Walmart Store 12,East Vincentfurt,Maine,1223,775878.18
13,Walmart Store 13,Chrisstad,South Carolina,1255,815177.17
23,Walmart Store 23,New Teresa,North Dakota,1321,834146.53
24,Walmart Store 24,West Matthew,Illinois,1195,748301.37
11,Walmart Store 11,Wichita,Kansas,1161,729061.21


In [0]:
def ask_walmart_lakehouse(question: str):
  q = question.lower()

  if "top stores" in q or "revenue" in q or "highest" in q:
    sql = """
            SELECT store_id, store_name, city, province, total_revenue, total_orders
            FROM walmart_project_catalog.gold.gold_store_sales_summary
            ORDER BY total_revenue DESC
            LIMIT 3
        """
  elif "orders" in q:
    sql = """
            SELECT store_id, store_name, city, province, total_orders
            FROM walmart_project_catalog.gold.gold_store_sales_summary
            ORDER BY total_orders DESC
            LIMIT 3
        """
  elif "province" in q or "state" in q or "illinois" in q:
    sql = """
            SELECT store_id, store_name, city, province, total_revenue
            FROM walmart_project_catalog.gold.gold_store_sales_summary
            WHERE province = 'Illinois'
        """
  else:
    sql = """
            SELECT * 
            FROM walmart_project_catalog.gold.gold_store_sales_summary 
            LIMIT 5
        """

  print(f"Executing Query:\n{sql.strip()}\n")
  display(spark.sql(sql))


# Test your zero-cost natural language function
ask_walmart_lakehouse("What are the top stores by revenue?")

Executing Query:
SELECT store_id, store_name, city, province, total_revenue, total_orders
            FROM walmart_project_catalog.gold.gold_store_sales_summary
            ORDER BY total_revenue DESC
            LIMIT 3



store_id,store_name,city,province,total_revenue,total_orders
10,Walmart Store 10,Nixonburgh,Oklahoma,840945.79,1295
23,Walmart Store 23,New Teresa,North Dakota,834146.53,1321
22,Walmart Store 22,Kristaberg,Minnesota,827663.99,1245
